In [1]:
import sys
sys.path.append('..')
from fastai.vision.all import *
from mocatml.utils import *
convert_uuids_to_indices()
from mocatml.data import *
from mocatml.models.conv_rnn import *
from mocatml.models.phy_original import *
from mocatml.models.seq2seq import TeacherForcing
from mygrad import sliding_window_view
from tsai.imports import my_setup
from tsai.utils import yaml2dict, dict2attrdict
from fastai.callback.schedule import valley, steep
from fastai.callback.wandb import WandbCallback
import wandb

In [2]:
!ls

PatchTST.ipynb		  data_gen.sh.log-25358349  phydnet.ipynb
TSTPlus.ipynb		  data_gen.sh.log-25358429  phydnet.py
config			  data_gen.sh.log-25358435  phydnet.sh
convgru.ipynb		  data_gen.sh.log-25375479  phydnet.sh.log-25425684
convgru_eval.ipynb	  data_gen.sh.log-25375484  phydnet.sh.log-25425749
convgru_optuna.ipynb	  data_gen.sh.log-25375489  phydnet.sh.log-25425800
data_exp.ipynb		  data_gen.sh.log-25375497  pm_tmp.ipynb
data_gen.py		  data_vis.ipynb	    pm_tmp.py
data_gen.sh		  data_vis.py		    resnet
data_gen.sh.log-25321306  eval.ipynb		    train_nb.ipynb
data_gen.sh.log-25321505  eval_result		    transformer.ipynb
data_gen.sh.log-25340237  exp.ipynb		    tsf.ipynb
data_gen.sh.log-25342239  lr_finder.ipynb	    wandb
data_gen.sh.log-25345145  models
data_gen.sh.log-25345198  optuna_study.ipynb


In [ ]:
from fastai.callback.schedule import LRFinder

@patch_to(LRFinder)
def after_fit(self):
    self.learn.opt.zero_grad() # Needed before detaching the optimizer for future fits
    tmp_f = self.path/self.model_dir/self.tmp_p/'_tmp.pth'
    if tmp_f.exists():
        self.learn.load(f'{self.tmp_p}/_tmp', with_opt=True, device='cpu')
        self.tmp_d.cleanup()

In [ ]:
my_setup()

In [ ]:
config_base = yaml2dict('./config/base.yaml', attrdict=True)
config_base.phdnet = yaml2dict('./config/phdnet/phydnet.yaml', attrdict=True)
#config = AttrDict({**config_base, **config_e2e})
config = AttrDict(config_base)
config

In [ ]:
default_device(0 if config.device == 'cpu' else config.device)

In [6]:
run = wandb.init(dir=ifnone(config.wandb.dir, '../'),
                 project=config.wandb.project, 
                 config=config,
                 group=config.wandb.group,
                 mode=config.wandb.mode, 
                 anonymous='never') if config.wandb.enabled else None
config = dict2attrdict(run.config) if config.wandb.enabled else config
config

```json
{ 'bs': 32,
  'data': { 'dataset': ['x2x2', 'x5x5', 'x10x10', 'x15x15'],
            'path': '~/mocat-ml/data/TLE_density_all_',
            'usage': [0.25, 0.5, 0.75, 1]},
  'device': 'cuda',
  'gap': 0,
  'horizon': 4,
  'lookback': 4,
  'lr_max': None,
  'mmap': True,
  'n_epoch': 20,
  'normalize': True,
  'num_workers': 0,
  'partial_loss': None,
  'phdnet': { 'attn': False,
              'blur': False,
              'coord_conv': False,
              'debug': False,
              'ks': 3,
              'n_in': 1,
              'n_out': 1,
              'norm': None,
              'rnn_ks': 5,
              'strategy': 'zero',
              'szs': [16, 64, 96]},
  'save_learner': True,
  'seed': None,
  'sel_steps': None,
  'stride': 8,
  'tmp_folder': 'tmp',
  'wandb': { 'dir': None,
             'enabled': True,
             'group': None,
             'log_learner': False,
             'mode': 'offline',
             'project': 'mocatml'}}
```

In [7]:
data = np.load(Path(config.data.path + config.data.dataset[0] + '.npy').expanduser(), 
               mmap_mode='c' if config.mmap else None)

data = data[:, :config.sel_steps]

data_sw = np.lib.stride_tricks.sliding_window_view(data, 
                                               config.lookback + config.horizon + config.gap, 
                                               axis=1)[:,::config.stride,:]
samples_per_simulation = data_sw.shape[1]
data_sw = data_sw.transpose(0,1,4,2,3)
data_sw = data_sw.reshape(-1, *data_sw.shape[2:])
data_sw.shape

(30400, 8, 36, 99)

In [8]:
data_sw = data_sw[:, :, :32, :32]

In [9]:
splits = RandomSplitter()(data)
ds = DensityData(data_sw, lbk=config.lookback, h=config.horizon, gap=config.gap)
train_idxs = calculate_sample_idxs(splits[0], samples_per_simulation)
valid_idxs = calculate_sample_idxs(splits[1], samples_per_simulation)

mocat_stats = (np.mean(data[splits[0]]), np.std(data[splits[0]]))

train_tl = TfmdLists(train_idxs, DensityTupleTransform(ds))
valid_tl = TfmdLists(valid_idxs, DensityTupleTransform(ds))
dls = DataLoaders.from_dsets(train_tl, valid_tl, bs=config.bs, device=default_device(),
                    after_batch=[Normalize.from_stats(*mocat_stats)] if \
                    config.normalize else None,
                    num_workers=config.num_workers)
    
#export
class PHyCallback(Callback):
    def after_pred(self):
        self.learn.pred, self.loss_phy = self.pred
    def after_loss(self):
        self.learn.loss += self.loss_phy
        
mse_loss = StackLoss(MSELossFlat(axis=1))
metrics = []

In [10]:
phycell =  PhyCell(input_shape=(16, 16), input_dim=64, F_hidden_dims=[49], n_layers=1, kernel_size=(7,7)) 
convlstm = ConvLSTM(input_shape=(16, 16), input_dim=64, hidden_dims=[128,128,64], n_layers=3, kernel_size=(3,3))   
encoder =  EncoderRNN(phycell, convlstm)

model = StackUnstack(PhyDNet(encoder, sigmoid=True, moment=True), dim=1).cuda()
cbs = L() + [ShowGraphCallback()] + [TeacherForcing(10), PHyCallback()]
learn = Learner(dls, model, loss_func=mse_loss, cbs=cbs, metrics=metrics, opt_func = ranger)
lr_max = learn.lr_find()
lr_max

layer  0 input dim  64  hidden dim  128
layer  1 input dim  128  hidden dim  128
layer  2 input dim  128  hidden dim  64


TypeError: leaky_relu(): argument 'input' (position 1) must be Tensor, not list

In [ ]:
# https://forums.fast.ai/t/fit-flat-cos-s-hyperparameters-meaning-and-tuning/85983/3
learn.fit_flat_cos(config.n_epoch, 1e-3)
# learn.fit_one_cycle(config.n_epoch, lr_max=lr_max)

In [ ]:
phycell1 =  PhyCell(input_shape=(16, 16), input_dim=64, F_hidden_dims=[49], n_layers=1, kernel_size=(7,7)) 
convlstm1 = ConvLSTM(input_shape=(16, 16), input_dim=64, hidden_dims=[128,128,64], n_layers=3, kernel_size=(3,3))   
encoder1 =  EncoderRNN(phycell1, convlstm1)

model1 = StackUnstack(PhyDNet(encoder1, sigmoid=True, moment=True), dim=1).cuda()
cbs = L() + [ShowGraphCallback()] + [TeacherForcing(10), PHyCallback()]
learn1 = Learner(dls, model1, loss_func=mse_loss, cbs=cbs, metrics=metrics, opt_func = ranger)
lr_max = learn1.lr_find()
lr_max

In [ ]:
learn1.fit_flat_cos(config.n_epoch, 5e-3)

In [ ]:
rec = learn.recorder
rec1 = learn1.recorder

skip_start = 5

ax=plt.gca()
log=1

if log:
    ax.loglog(list(range(skip_start, len(rec.losses))), rec.losses[skip_start:], label='train 1e-3')
    ax.loglog(list(range(skip_start, len(rec1.losses))), rec1.losses[skip_start:], label='train 1e-4')
else:
    ax.plot(list(range(skip_start, len(rec.losses))), rec.losses[skip_start:], label='train 1e-3')
    ax.plot(list(range(skip_start, len(rec1.losses))), rec1.losses[skip_start:], label='train 1e-4')

ax.set_ylabel('loss')
ax.set_xlabel('steps')
ax.set_title('learning curve')

idx = (np.array(rec.iters)<skip_start).sum()
valid_col = rec.metric_names.index('valid_loss') - 1 
ax.plot(rec.iters[idx:], L(rec.values[idx:]).itemgot(valid_col), label='valid 1e-3')

idx = (np.array(rec1.iters)<skip_start).sum()
valid_col = rec1.metric_names.index('valid_loss') - 1 
ax.plot(rec1.iters[idx:], L(rec1.values[idx:]).itemgot(valid_col), label='valid 1e-4')

ax.legend()

In [ ]:
def get_n_params(model):
    pp=0
    for p in list(model.parameters()):
        nn=1
        for s in list(p.size()):
            nn = nn*s
        pp += nn
    return pp

get_n_params(model)